# 📋 [SEC08_M07] 다수 방향 동시 드론 침입 및 통신 마비 복합 위협 상황 모의 스트레스 훈련
### AI Co-pilot 협업 기반 군사시설 방호 표준운용절차(SOP) 및 모의훈련 실습 (학생 실습 워크북)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/SEC08/sec08_m07_student.ipynb)

---

## 📌 프로젝트 개요
- **과정명:** 군사시설 드론 침입 경보 대응 모의훈련 (Level 3)
- **미션 번호:** SEC08_M07
- **미션 주제:** 다수 방향 동시 드론 침입 및 통신 마비 복합 위협 상황 모의 스트레스 훈련
- **운용 원칙:**
  1. **HITL 원칙:** AI 제안은 참고 지표이며, 최종 지휘·승인·교전 판단은 인간이 확정한다.
  2. **군사보안 준수:** 실제 부대명, 좌표, 통신 음어, 무기 제원은 일체 사용하지 않으며 가상 시뮬레이션 데이터만 활용한다.
  3. **안전 제일 원칙:** 실사격/장비 조작은 배제하며 구두 선언과 모의 기동으로만 진행하고, 잔해물 접촉은 엄격히 금지한다.


In [ ]:
# [SEC08] 군사시설 드론 침입 대응 모의훈련 자동 분석 및 검증 헬퍼 모듈
import pandas as pd
import numpy as np
import json
import os

# 8대 표준 산출물 디렉토리 자동 생성
dirs = ['docs', 'sop', 'scenarios', 'logs', 'reports', 'aar', 'eval', 'package']
for d in dirs:
    os.makedirs(d, exist_ok=True)
print('📁 [표준 디렉토리 생성] 8대 산출물 폴더 준비 완료 (docs/, sop/, scenarios/, logs/, reports/, aar/, eval/, package/)')

def verify_mission1_raci(raci_data, checklist_items):
    print('=' * 65)
    print('📋 [미션 1 검증] 7단계 부서별 RACI 매트릭스 & 체크리스트 검증')
    print('=' * 65)
    stages = ['접수', '식별', '경보 발령', '전파', '방호 조치', '대응·통제', '해제·인계']
    stage_pass = True
    for s in stages:
        if s not in raci_data:
            print(f'❌ 누락된 단계: {s}')
            stage_pass = False
        else:
            roles = raci_data[s]
            acc_count = sum(1 for r, duty in roles.items() if '책' in duty or 'A' in duty)
            if acc_count != 1:
                print(f'⚠️ [{s} 단계] 최종 책임자(Accountable/책)가 1명이 아닙니다! (현재 {acc_count}명)')
                stage_pass = False
    check_count = len(checklist_items)
    check_pass = check_count >= 15
    print(f'✔ 현장 체크리스트 항목 수: {check_count}개 / 기준 15개 이상 -> {"[PASS]" if check_pass else "[FAIL]"}')
    print(f'✔ 7단계 RACI 매트릭스 책임자 무결성: {"[PASS]" if stage_pass else "[FAIL]"}')
    if stage_pass and check_pass:
        print('\n🎉 [미션 1 PASS] 기지 방호 4개 부서 역할 분담과 체크리스트가 전술 기준에 부합합니다!')
    else:
        print('\n❌ [보완 필요] 누락된 단계나 중복된 책임자를 확인하여 수정하십시오.')

def verify_mission2_sop_checklist(sop_stages, safety_rules, pocket_guide):
    print('=' * 65)
    print('📋 [미션 2 검증] 3단계 방호 등급, 8대 필수 안전수칙 & 포켓가이드 검증')
    print('=' * 65)
    req_stages = ['경계', '경보', '비상']
    stage_ok = all(s in sop_stages for s in req_stages)
    safety_ok = len(safety_rules) >= 8
    pocket_ok = len(pocket_guide) >= 3 and any('승인' in str(v) for v in pocket_guide.values())
    print(f'✔ 3단계 방호 등급(경계/경보/비상) 완비: {"[PASS]" if stage_ok else "[FAIL]"}')
    print(f'✔ 필수 안전/보안 조치(S1~S8) 8개 이상 배치: {len(safety_rules)}개 -> {"[PASS]" if safety_ok else "[FAIL]"}')
    print(f'✔ 1-Page 휴대용 요약 카드(Pocket Guide) 가독성 규격: {"[PASS]" if pocket_ok else "[FAIL]"}')
    if stage_ok and safety_ok and pocket_ok:
        print('\n🎉 [미션 2 PASS] 시설별 방호 단계 및 부서별 행동 요령 체크리스트가 기준을 충족합니다!')
    else:
        print('\n❌ [보완 필요] 8대 안전수칙 배치 및 지휘관 승인선을 재확인하십시오.')

def verify_mission3_msel(msel_events, report_timelines):
    print('=' * 65)
    print('📋 [미션 3 검증] 3자 전파 체계 시간 기준 및 MSEL 시나리오 검증')
    print('=' * 65)
    event_cnt = len(msel_events)
    normal_cnt = sum(1 for e in msel_events if e.get('type') == '정상')
    abnormal_cnt = sum(1 for e in msel_events if e.get('type') == '비정상')
    has_emergency = any(e.get('is_unexpected', False) for e in msel_events)
    timeline_ok = len(report_timelines) >= 5
    print(f'✔ 주입 사건 수: {event_cnt}건 (기준: 10건 이상) -> {"[PASS]" if event_cnt >= 10 else "[FAIL]"}')
    print(f'✔ 정상:비정상 비율: {normal_cnt}:{abnormal_cnt} (기준: 6:4 권장) -> {"[PASS]" if normal_cnt >= 5 and abnormal_cnt >= 3 else "[FAIL]"}')
    print(f'✔ 돌발 변수(통신 두절·센서 오류 등) 1건 이상 포함: {"[PASS]" if has_emergency else "[FAIL]"}')
    print(f'✔ 3자 전파 체계 5개 구간 시간 기준 수립: {"[PASS]" if timeline_ok else "[FAIL]"}')
    if event_cnt >= 10 and has_emergency and timeline_ok:
        print('\n🎉 [미션 3 PASS] 전파 체계 및 MSEL 주입 시나리오가 실전 모의훈련에 적합합니다!')
    else:
        print('\n❌ [보완 필요] 사건 건수(10건) 및 돌발 변수 요건을 보강하십시오.')

def analyze_mission4_dryrun(log_df):
    print('=' * 65)
    print('⏱️ [미션 4 분석] 1차 Dry-run 훈련 타임라인 및 SOP 준수율 계산')
    print('=' * 65)
    avg_lead_time = log_df['lead_time_sec'].mean()
    sop_pass_rate = (log_df['sop_pass'].sum() / len(log_df)) * 100
    print(log_df[['event_id', 'detect_sec', 'report_sec', 'relay_sec', 'action_sec', 'lead_time_sec', 'sop_pass']])
    print('-' * 65)
    print(f'✔ 평균 조치 완료 시간: {avg_lead_time:.1f}초 (기준: 180초 이내) -> {"[PASS]" if avg_lead_time <= 180 else "[FAIL]"}')
    print(f'✔ SOP 단계별 준수율: {sop_pass_rate:.1f}% (기준: 80% 이상) -> {"[PASS]" if sop_pass_rate >= 80 else "[FAIL]"}')
    if avg_lead_time <= 180 and sop_pass_rate >= 80:
        print('\n🎉 [미션 4 PASS] 1차 Dry-run 성과 목표를 달성했습니다! 상위 병목 구간을 AAR에 인계하십시오.')

def grade_mission5_cross_verification(user_answers, answer_key):
    print('=' * 65)
    print('🎯 [미션 5 채점] 오류 주입 교차확인 판정 결과 검증')
    print('=' * 65)
    correct = 0
    miss_threat = 0
    total = len(answer_key)
    for card, info in answer_key.items():
        user_pred = user_answers.get(card, {}).get('pred', '')
        truth = info['truth']
        is_threat = ('실제 위협' in truth or '실제 위험' in truth)
        if user_pred.strip() == truth.strip():
            correct += 1
        elif is_threat and ('오경보' in str(user_pred)):
            miss_threat += 1
            print(f'🚨 [치명적 오류] {card} 위협을 오경보로 오판했습니다! (Miss 발생)')
    success_rate = (correct / total) * 100
    print(f'✔ 허위 경보/기만 식별 성공률: {success_rate:.1f}% ({correct}/{total}) (기준: 85% 이상)')
    print(f'✔ 실제 위협 미탐(Miss) 건수: {miss_threat}건 (기준: 0건)')
    if success_rate >= 85.0 and miss_threat == 0:
        print('\n🎉 [미션 5 PASS] 교차 확인을 통해 적 기만 및 허위 경보를 성공적으로 필터링했습니다!')

def verify_mission6_eod_handover(sbar_data, quiz_score, contact_count):
    print('=' * 65)
    print('💣 [미션 6 검증] 드론 잔해물 격리 및 EOD 인계 프로토콜 검증')
    print('=' * 65)
    req_sbar = ['S', 'B', 'A', 'R']
    sbar_ok = all(k in sbar_data for k in req_sbar)
    quiz_ok = quiz_score >= 4
    contact_ok = contact_count == 0
    print(f'✔ SBAR 4단계 필수 항목 완비: {"[PASS]" if sbar_ok else "[FAIL]"}')
    print(f'✔ 돌발 퀴즈 점수: {quiz_score}/5점 (기준: 4점 이상) -> {"[PASS]" if quiz_ok else "[FAIL]"}')
    print(f'✔ 잔해물 무단 접근/접촉 위반 건수: {contact_count}건 (기준: 0건) -> {"[PASS]" if contact_ok else "[FAIL]"}')
    if sbar_ok and quiz_ok and contact_ok:
        print('\n🎉 [미션 6 PASS] EOD 안전 인계 및 현장 격리 절차가 완벽히 준수되었습니다!')
    else:
        print('\n❌ [보완 필요] 무접촉 원칙 및 SBAR 인계 서식을 점검하십시오.')

def verify_mission7_stress_drill(stress_log):
    print('=' * 65)
    print('⚡ [미션 7 검증] 다방향 동시 침입 & 통신 마비 스트레스 훈련 검증')
    print('=' * 65)
    trans_time = stress_log.get('transition_time_sec', 999)
    action_success = stress_log.get('action_success_rate', 0.0)
    delegation_time = stress_log.get('delegation_time_sec', 999)
    safety_violations = stress_log.get('safety_violations', 99)
    print(f'✔ 긴급 모드 전환 시간: {trans_time}초 (기준: 60초 이내) -> {"[PASS]" if trans_time <= 60 else "[FAIL]"}')
    print(f'✔ 10대 위험 방호 조치 성공률: {action_success:.1f}% (기준: 90% 이상) -> {"[PASS]" if action_success >= 90 else "[FAIL]"}')
    print(f'✔ 임무형 지휘 위임 소요 시간: {delegation_time}초 (기준: 60초 이내) -> {"[PASS]" if delegation_time <= 60 else "[FAIL]"}')
    print(f'✔ 야간 안전 통제 위반: {safety_violations}건 (기준: 0건) -> {"[PASS]" if safety_violations == 0 else "[FAIL]"}')
    if trans_time <= 60 and action_success >= 90 and delegation_time <= 60 and safety_violations == 0:
        print('\n🎉 [미션 7 PASS] 극한 스트레스 상황에서도 조직의 지휘 복원력과 방호 태세가 유지되었습니다!')
    else:
        print('\n❌ [보완 필요] 전환 시간 단축 및 지휘 위임 절차를 재훈련하십시오.')

def verify_mission8_aar_rca(fact_checks, rca_matrix, improvements):
    print('=' * 65)
    print('🔍 [미션 8 검증] 사후검토(AAR) 팩트체크 및 근본 원인 분석(RCA) 검증')
    print('=' * 65)
    fc_cnt = len(fact_checks)
    rca_cats = ['절차', '도구', '환경', '숙련']
    rca_ok = all(any(c in item.get('cat', '') for item in rca_matrix) for c in rca_cats)
    imp_cnt = len(improvements)
    print(f'✔ 진술 vs 로그 팩트체크 식별 건수: {fc_cnt}건 (기준: 5건 이상) -> {"[PASS]" if fc_cnt >= 5 else "[FAIL]"}')
    print(f'✔ 4대 축(절차·도구·환경·숙련) RCA 매트릭스 분류: {"[PASS]" if rca_ok else "[FAIL]"}')
    print(f'✔ 사실 기반 제도적 개선점 도출: {imp_cnt}건 (기준: 5건 이상) -> {"[PASS]" if imp_cnt >= 5 else "[FAIL]"}')
    if fc_cnt >= 5 and rca_ok and imp_cnt >= 5:
        print('\n🎉 [미션 8 PASS] 비난 배제 팩트 중심 AAR을 통해 실효적 SOP 개정안이 도출되었습니다!')
    else:
        print('\n❌ [보완 필요] 4대 요인 매트릭스 빈칸 및 개선 과제를 확충하십시오.')

def compare_mission9_retrial(m4_data, m9_data):
    print('=' * 65)
    print('📈 [미션 9 성과 분석] 1차(미션4) 대비 2차 재훈련 반응 속도 & 오류 개선율')
    print('=' * 65)
    time1 = m4_data['avg_time']
    time2 = m9_data['avg_time']
    time_reduction = ((time1 - time2) / time1) * 100
    err1 = m4_data['errors']
    err2 = m9_data['errors']
    err_reduction = ((err1 - err2) / err1) * 100
    print(f'1. 전체 조치 소요 시간: {time1:.1f}초 ➔ {time2:.1f}초 (단축률: {time_reduction:.1f}% / 목표 30% 이상)')
    print(f'2. SOP 절차 오류 건수: {err1}건 ➔ {err2}건 (감소율: {err_reduction:.1f}% / 목표 50% 이상)')
    print('-' * 65)
    pass_time = time_reduction >= 30.0
    pass_err = err_reduction >= 50.0
    if pass_time and pass_err:
        print('\n🎉 [미션 9 PASS] 재훈련 성과 지표(시간 30% 단축, 오류 50% 감소)를 완벽히 달성했습니다!')

def verify_mission10_sop_package(package_path, chapters):
    print('=' * 65)
    print('📘 [미션 10 검증] 군사시설 드론 대응 5개 장 표준 훈련 패키지 검증')
    print('=' * 65)
    file_ok = os.path.exists(package_path) and os.path.getsize(package_path) > 100
    chap_ok = len(chapters) >= 5
    print(f'✔ 5개 장(개요-SOP-시나리오-루브릭-AAR) 구성: {len(chapters)}개 장 -> {"[PASS]" if chap_ok else "[FAIL]"}')
    print(f'✔ 최종 훈련 패키지 파일({package_path}) 생성: {"[PASS]" if file_ok else "[FAIL]"}')
    if file_ok and chap_ok:
        print('\n🎉 [미션 10 PASS] 야전 교관이 즉각 활용 가능한 완성형 훈련 패키지가 구축되었습니다!')
    else:
        print('\n❌ [보완 필요] 파일 생성 경로 및 5개 장 구성을 확인하십시오.')

print('✔ [헬퍼 모듈 로드 완료] SEC08 전체 10개 미션 자동 검증 및 분석 함수 준비 완료')

---

## 🎯 미션 07. 다수 방향 동시 드론 침입 및 통신 마비 복합 위협 상황 모의 스트레스 훈련

### 📋 미션 개요 및 산출물
- **필수 산출물:** `logs/sec08_stress_drill_results.md`
- **핵심 활동:** 가상 시나리오 분석 및 데이터 검증 실행

### 🤖 AI Co-pilot 협업 가이드 (프롬프트 질의 및 검증)
> **💡 프롬프트 질의 예시:**
> *"야간 다방향 동시 침입 및 통신 두절 스트레스 훈련에서 긴급 모드 전환 속도와 임무형 지휘 위임 성공률 평가표를 작성해줘."*

### 💻 [학생 실습 및 검증 코드]


In [ ]:
# [미션 7 실습 코드] TODO: 스트레스 훈련 실측 데이터(stress_log)를 입력하고 검증하세요.
stress_log = {
    'transition_time_sec': 55,
    'action_success_rate': 92.0,
    'delegation_time_sec': 50,
    'safety_violations': 0
}

verify_mission7_stress_drill(stress_log)
